# Tutorial 170 — Conversation memory and verified historical recall

[Tutorial 160](./160_project_types.ipynb) introduced project types and
intents. Here we examine **project-local conversation memory**: which
questions and replies can be indexed, how Save/Like/Dislike and explicit
exclusions change eligibility, and how a historical request recovers actual
saved messages without treating them as scientific evidence.

This is a **model-free, offline demonstration** using KíLín's real conversation
and feedback persistence APIs and a deterministic stand-in embedding model.
It does not call a chat model or download model weights. The usual notebook
setup helper may install missing Python dependencies if your environment has
not already been prepared.

The fictional discussion below is solely a teaching fixture. The assistant's
recorded words are *not* scientific observations or approved decisions.

[tutorial guide](./README.md)


## 1. Prepare the local checkout

Use the Python 3.12 environment from Tutorial 10. The same setup cells as
other notebooks locate the checkout and confirm its dependencies.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "kilin").is_dir():
    root = root.parent
if not (root / "kilin").is_dir():
    raise FileNotFoundError("Open the notebook from the KíLín checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 2. Inspect a project's memory policy

A type enables or disables long-term memory independently of source
retrieval. By default, **Brainstorming (Creative)** indexes eligible user questions,
but assistant answers require explicit Save or Like. `max_hits` and
`max_tokens` bound the separate memory context. The selected intent can
also forbid indexing or reading memory, regardless of these global values.


In [ ]:
from kilin.project_type import project_type

config = project_type("creative_brainstorming")
policy = config.conversation_memory
assert policy.enabled and policy.questions == "all"
assert policy.answers == "selected"
print("Conversation memory:", policy)
print("Intent indexing:", config.intents["open_ideation"].memory_indexing)
print("Intent retrieval:", config.intents["open_ideation"].memory_read)


## 3. Create a private, fictional saved conversation

Ordinary KíLín conversations are created by the GUI or `RAGSession`.
For a **teaching fixture only**, we construct two `ConversationTurn` records
and write them using the same `ConversationStore` and `TraceLogger` storage
interfaces. The example chooses a **new** numbered workspace in `sandbox/`
and never overwrites an earlier run.

Each saved question and its answer share a stable turn UUID. That identity,
not text similarity or timestamps, associates the reply with the question.


In [ ]:
from dataclasses import asdict
from kilin.conversations import ConversationStore
from kilin.rag import ConversationTurn
from kilin.trace_logging import TraceConfig, TraceLogger

parent = root / "sandbox"
parent.mkdir(parents=True, exist_ok=True)
workspace = parent / "tutorial_170_conversation_memory"
suffix = 2
while workspace.exists():
    workspace = parent / f"tutorial_170_conversation_memory_{suffix}"
    suffix += 1
workspace.mkdir()
(workspace / "indexes").mkdir()
store = ConversationStore(workspace)
conversation = store.create()
print("Example workspace:", workspace)
print("Conversation ID:", conversation.conversation_id)


In [ ]:
turns = [
    ConversationTurn(
        timestamp="2026-11-15T12:00:00+00:00",
        question="Did we consider optical tweezers for force sensing?",
        answer="We proposed an initial calibration experiment.",
        evidence=(),
        parameters={"intent": "open_ideation"},
    ),
    ConversationTurn(
        timestamp="2026-11-15T12:10:00+00:00",
        question="Could microfluidic mixing affect protein folding?",
        answer="One speculative mechanism is intermittent mixing.",
        evidence=(),
        parameters={"intent": "open_ideation"},
    ),
]
path = store.state_path(conversation)
logger = TraceLogger(
    TraceConfig(enabled=True, output_path=path),
    default_directory=path.parent,
)
try:
    logger.update_conversation({"turns": [
        asdict(turn) for turn in turns
    ]})
finally:
    logger.close()
print("Saved checkpoint:", path.relative_to(workspace))


## 4. Synchronize the derived index without a downloaded model

In an ordinary running project, KíLín uses its already-loaded embedding
model. This tiny stand-in supports the same two embedding methods but returns
*illustrative vectors*. It is **not** a meaningful semantic model, and its
scores must not be used to evaluate retrieval quality.

Explicit historical lookup instead uses lexical matching of saved records
and **does not invoke** `embed_query`.


In [ ]:
from kilin.conversation_memory.index import synchronize_memory


class TeachingEmbedding:
    """Provide synthetic vectors for a deterministic storage example."""

    def __init__(self):
        self.documents = []
        self.queries = []

    @staticmethod
    def vector(text):
        return ([1.0, 0.0] if "tweezers" in text.lower()
                else [0.0, 1.0])

    def embed_documents(self, texts, *, purpose="documents"):
        self.documents.extend(texts)
        return [self.vector(text) for text in texts]

    def embed_query(self, text):
        self.queries.append(text)
        return self.vector(text)


embedding = TeachingEmbedding()


def sync():
    """Refresh the project's independently persisted memory index."""

    return synchronize_memory(
        workspace, config, embedding,
        model_identity="synthetic-teaching-embedding-v1",
    )


index = sync()
print("Indexed entries:", [(r.role, r.text) for r in index.records])
assert {record.role for record in index.records} == {"user"}
assert len(index.records) == 2


## 5. Save and Like a reply independently

`FeedbackStore` is authoritative user-owned state under
`memory/feedback.json`. A Save and a Like are separate choices; either
makes an eligible answer selectable when `answers: selected`.
Feedback applies to **one message role**, identified by the turn UUID.
This example selects the first assistant reply, without marking its user
question or the other answer.


In [ ]:
from kilin.conversation_memory.feedback import FeedbackStore

feedback = FeedbackStore(workspace)
first_uuid = turns[0].conversation_turn_uuid
before = feedback.load()
after = feedback.update(
    conversation.conversation_id, first_uuid, "assistant",
    saved=True, reaction="liked",
    expected_revision=before.revision,
)
rating = after.get(first_uuid, "assistant")
assert rating.saved and rating.reaction == "liked"
index = sync()
assert [r.role for r in index.records].count("assistant") == 1
print("Selected answer:", rating)
print("New embeddings:", index.embedded_count)


## 6. Verify historical recall from saved messages

A question such as *“What did we decide about optical tweezers?”* selects
the conservative historical-lookup path. The result can establish which
messages were saved and which reply followed a matching question, **not**
that the experiment was approved or the scientific claim validated.

PR12 keeps eligible question–answer pairs adjacent within the configured
hit and token budgets. Memory labels `M1`, `M2`, etc. identify saved
conversations; they are **not** citations to scientific documents.


In [ ]:
from kilin.conversation_memory.recall import recall_topic
from kilin.conversation_memory.retrieval import (
    render_memory_hits, search_memory,
)

historical_question = "What did we decide about optical tweezers?"
topic = recall_topic(historical_question)
assert topic == "optical tweezers"
hits = search_memory(
    historical_question, index, embedding,
    max_hits=policy.max_hits, max_tokens=policy.max_tokens,
    recall_topic=topic,
)
assert [hit.record.role for hit in hits] == ["user", "assistant"]
assert hits[0].record.turn_uuid == hits[1].record.turn_uuid
assert not embedding.queries
print(render_memory_hits(hits))


In [ ]:
one_hit = search_memory(
    historical_question, index, embedding,
    max_hits=1, max_tokens=policy.max_tokens,
    recall_topic=topic,
)
assert [hit.record.role for hit in one_hit] == ["user"]
print("With max_hits=1:", one_hit[0].text)


## 7. Dislike a reply without erasing its question

Dislike disqualifies the assistant reply from long-term memory even if it
remains Saved. The original saved conversation and independent user
question remain intact. The default `history.exclude_disliked_answers`
also applies to **ordinary chat-history context**, a separate policy from
long-term indexing. Resetting the reaction restores eligibility.


In [ ]:
feedback.update(
    conversation.conversation_id, first_uuid, "assistant",
    saved=None, reaction="disliked",
)
index = sync()
hits = search_memory(
    historical_question, index, embedding,
    max_hits=policy.max_hits, max_tokens=policy.max_tokens,
    recall_topic=topic,
)
assert [hit.record.role for hit in hits] == ["user"]
print("With the reply disliked:", [hit.text for hit in hits])


In [ ]:
restored = feedback.update(
    conversation.conversation_id, first_uuid, "assistant",
    saved=None, reaction="neutral",
)
assert restored.get(first_uuid, "assistant").saved
index = sync()
assert any(r.turn_uuid == first_uuid and r.role == "assistant"
           for r in index.records)
print("Answer restored by resetting Dislike; Save remains selected.")


## 8. Explicitly exclude and restore a question

In **Settings → Memory → Manage remembered messages**, Exclude/Restore
controls *retrieval eligibility*, not deletion of the original conversation.
The core exposes the same operation through `FeedbackStore`, with an optional
revision check to reject stale concurrent edits. Unlike turning Save off,
explicit exclusion also overrides automatic question indexing.


In [ ]:
state = feedback.load()
feedback.update(
    conversation.conversation_id, first_uuid, "user",
    saved=None, reaction=None, excluded=True,
    expected_revision=state.revision,
)
index = sync()
assert not any(r.turn_uuid == first_uuid and r.role == "user"
               for r in index.records)
print("First question explicitly excluded from the index.")


In [ ]:
feedback.update(
    conversation.conversation_id, first_uuid, "user",
    saved=None, reaction=None, excluded=False,
)
index = sync()
assert any(r.turn_uuid == first_uuid and r.role == "user"
           for r in index.records)
print("First question restored without deleting its saved turn.")


## 9. Inspect branch provenance and rebuildable state

Branches can share the same original turn UUID. Index records retain all
**active origins** without duplicating the underlying message. Archiving
a branch drops only that origin on the next synchronization. The derived
`indexes/conversation_memory.json` can be rebuilt from eligible saved
turns and the authoritative `memory/feedback.json`.


In [ ]:
branch = store.create(
    branch_from=conversation.conversation_id, turn_count=1,
)
index = sync()
first_question = next(
    r for r in index.records
    if r.turn_uuid == first_uuid and r.role == "user"
)
assert len(first_question.origins) == 2
print("Shared origins:", first_question.origins)
store.archive(branch.conversation_id)
index = sync()
first_question = next(
    r for r in index.records
    if r.turn_uuid == first_uuid and r.role == "user"
)
assert len(first_question.origins) == 1
print("After branch archive:", first_question.origins)


## 10. Apply the distinction in a real project

- **Questions** may be indexed automatically. **Answers** are eligible only
  when the saved project and originating intent policies permit them.
- **Save** and **Like** select answers; **Dislike** and explicit **Exclude**
  override selection. Restoring a message does not alter its original text.
- Historical questions verify eligible *saved discussion*, not a research
  finding or the researcher's acceptance of an answer. If no matching eligible
  message exists, KíLín must report that it **cannot verify** the discussion,
  not that it never happened.
- The Memory tab manages indexing, inspection, restoration and manual Update.
  The Intents tab configures per-intent memory indexing and reading.
- Remembered messages have `Memory M1`, `M2`, etc. references, distinct from
  numbered **primary-source citations**. Reviewing still leaves all
  assessments and judgments to the researcher.

The example stays in `sandbox/` for inspection. Do not publish its raw traces
as sanitized data; in real projects they may contain private conversations,
model output, prompts, and local paths. The artificial embeddings here are
not a substitute for the real semantic retrieval benchmark.

For the authoritative details, see
[Conversation memory](../docs/conversation_memory.md) and
[Configuration](../docs/configuration.md).


## Continue the tutorial series

**Previous:** [Tutorial 160 — Define project types and route intents](./160_project_types.ipynb)

**Next:** [Tutorial 180 — Create a custom KíLín tool](./180_create_custom_tool.ipynb)

Tutorial 180 extends project behavior with a concrete tool implementation.
